<img src="https://hilpisch.com/tpq_logo_bic.png" width="20%" align="right">

# Python for Finance, 3rd Edition
## Appendix C · Numerical Methods and Simulation Notes

&copy; Dr. Yves J. Hilpisch<br>
AI-supported by GPT 5.x<br>
The Python Quants GmbH | https://tpq.io<br>
https://hilpisch.com | https://linktr.ee/dyjh

## Notebook Goals
This notebook mirrors the appendix examples in a Colab-ready format so that you can run, tweak, and extend them interactively.

### How to Use This Notebook
- Run the cells top to bottom the first time to create all variables.
- Use additional cells for your own experiments or GenAI-assisted refactorings.
- Refer back to the book text for detailed explanations and context.

## Root Finding and Implied Quantities
A root-finding problem solves latexmath:[f(x)=0].

In [ ]:
import numpy as np

In [ ]:
from math import exp, log, sqrt

In [ ]:
from scipy import optimize, stats

In [ ]:
def bs_call(s, k, r, sigma, t):
    d1 = (log(s / k) + (r + 0.5 * sigma**2) * t) / (sigma * sqrt(t))
    d2 = d1 - sigma * sqrt(t)
    return s * stats.norm.cdf(d1) - k * exp(-r * t) * stats.norm.cdf(d2)

In [ ]:
market = 9.4134

In [ ]:
def error(vol):
    return bs_call(100.0, 100.0, 0.03, vol, 1.0) - market

In [ ]:
iv = optimize.brentq(error, 0.01, 1.0)

In [ ]:
round(iv, 4), round(bs_call(100, 100, 0.03, iv, 1), 4)

## Numerical Integration
Numerical integration approximates integrals that cannot be evaluated easily in
closed form.

In [ ]:
from scipy import integrate

In [ ]:
val, err = integrate.quad(
    lambda x: stats.norm.pdf(x),
    -1.96,
    1.96,
)

In [ ]:
round(val, 6), f"{err:.1e}"

## Interpolation of Curves and Surfaces
Market data arrive at discrete points: maturities on a yield curve, strikes on
a volatility smile, or expiries on a futures curve.

In [ ]:
from scipy import interpolate

In [ ]:
t = np.array([0.25, 0.5, 1.0, 2.0, 5.0])

In [ ]:
zr = np.array([0.025, 0.027, 0.030, 0.033, 0.036])

In [ ]:
curve = interpolate.PchipInterpolator(t, zr)

In [ ]:
np.array([curve(0.75), curve(3.0)]).round(4)

## Finite Differences and Sensitivities
Finite differences approximate derivatives by revaluing a function at bumped
inputs.

In [ ]:
def call_delta(s, k, r, sigma, t):
    d1 = (log(s / k) + (r + 0.5 * sigma**2) * t) / (sigma * sqrt(t))
    return stats.norm.cdf(d1)

In [ ]:
h = 0.10

In [ ]:
up = bs_call(100 + h, 100, 0.03, 0.2, 1)

In [ ]:
down = bs_call(100 - h, 100, 0.03, 0.2, 1)

In [ ]:
fd_delta = (up - down) / (2 * h)

In [ ]:
round(fd_delta, 5), round(call_delta(100, 100, 0.03, 0.2, 1), 5)

## Monte Carlo Estimators and Standard Errors
A Monte Carlo estimator approximates an expectation by a sample average.

In [ ]:
rng = np.random.default_rng(20)

In [ ]:
z = rng.standard_normal(100_000)

In [ ]:
st = 100 * np.exp((0.03 - 0.5 * 0.2**2) + 0.2 * z)

In [ ]:
payoff = np.exp(-0.03) * np.maximum(st - 100, 0)

In [ ]:
price = float(payoff.mean())

In [ ]:
se = float(payoff.std(ddof=1) / np.sqrt(payoff.size))

In [ ]:
ci = np.array([price - 1.96 * se, price + 1.96 * se])

In [ ]:
round(price, 4), round(se, 4), ci.round(4)

## Variance Reduction
Variance reduction tries to lower estimator variance without changing the
quantity being estimated.

In [ ]:
rng = np.random.default_rng(21)

In [ ]:
z = rng.standard_normal(50_000)

In [ ]:
def mc_from_z(z):
    st = 100 * np.exp((0.03 - 0.5 * 0.2**2) + 0.2 * z)
    return np.exp(-0.03) * np.maximum(st - 100, 0)

In [ ]:
plain = mc_from_z(z)

In [ ]:
antithetic = 0.5 * (mc_from_z(z) + mc_from_z(-z))

In [ ]:
round(float(plain.std(ddof=1)), 4), round(float(antithetic.std(ddof=1)), 4)

## Time Discretization
Some stochastic differential equations have exact simulation schemes, but many
models used in finance do not.

In [ ]:
rng = np.random.default_rng(22)

In [ ]:
steps, paths = 252, 50_000

In [ ]:
dt = 1 / steps

In [ ]:
z = rng.standard_normal((paths, steps))

In [ ]:
s_euler = np.full(paths, 100.0)

In [ ]:
for j in range(steps):
    s_euler *= 1 + 0.03 * dt + 0.2 * np.sqrt(dt) * z[:, j]

In [ ]:
w_t = np.sqrt(dt) * z.sum(axis=1)

In [ ]:
s_exact = 100 * np.exp((0.03 - 0.5 * 0.2**2) + 0.2 * w_t)

In [ ]:
round(float(np.mean(np.abs(s_euler - s_exact))), 4)

<img src="https://hilpisch.com/tpq_logo_bic.png" width="20%" align="right">